# ITAI 1371 Final Project — Binary Classification

**Student:** Richard Scott  
**Project:** Classify network traffic as **Benign** or **Malicious**  
**Dataset:** `conn.log.labeled`

This notebook starts from the original labeled dataset and builds a clean Final project workflow:

1. Load and inspect the original dataset  
2. Preprocess the data  
3. Engineer useful features  
4. Split into Train / Validation / Test sets: 70% / 15% / 15%  
5. Train the required classification models  
6. Compare models using classification metrics  
7. Build a Voting Classifier using the best 3 models  
8. Compare against a Bayesian model using Gaussian Naive Bayes  
9. Export comparison tables and cleaned data files

> Note: The assignment mentions a "Voting Regressor," but this is a classification problem because the target is Benign vs Malicious. Therefore, this notebook uses a **Voting Classifier**.


## 1. Import Libraries

In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 2. Load the Dataset

Place `conn.log.labeled` in the same folder as this notebook.

If you are running in Google Colab, upload `conn.log.labeled` when prompted.


In [2]:
DATA_PATH = "conn.log.labeled"

# Colab-friendly upload fallback
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Dataset not found. Please upload conn.log.labeled.")
        uploaded = files.upload()
        if DATA_PATH not in uploaded:
            DATA_PATH = list(uploaded.keys())[0]
            print(f"Using uploaded file: {DATA_PATH}")
    except Exception:
        raise FileNotFoundError(
            "Could not find conn.log.labeled. Place the file in the same folder as this notebook."
        )

print(f"Using dataset: {DATA_PATH}")


Dataset not found. Please upload conn.log.labeled.


Saving conn.log.labeled to conn.log.labeled
Using dataset: conn.log.labeled


### Parse the Zeek-style log file

In [3]:
def load_zeek_conn_log(path):
    """Load the CTU Zeek conn.log.labeled file into a pandas DataFrame."""
    fields_line = None

    with open(path, "r", errors="replace") as f:
        for line in f:
            if line.startswith("#fields"):
                fields_line = line.strip()
                break

    if fields_line is None:
        raise ValueError("Could not find #fields line in the dataset.")

    fields_raw = fields_line.replace("#fields\t", "")
    columns = re.split(r"\t| {3,}", fields_raw)

    df = pd.read_csv(
        path,
        comment="#",
        sep=r"\t| {3,}",
        names=columns,
        engine="python",
        na_values=["-", "(empty)", ""]
    )

    df.columns = df.columns.str.strip()

    # Clean whitespace in text columns
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": np.nan, "None": np.nan})

    return df

df_raw = load_zeek_conn_log(DATA_PATH)

print("Dataset shape:", df_raw.shape)
display(df_raw.head())


Dataset shape: (10403, 23)


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,label,detailed-label
0,1.533043e+09,C5JLGOoxIw2dBZt47,192.168.100.113,123,81.2.254.224,123,udp,NaN,0.005490,48.0,...,NaN,0,Dd,1,76,1,76,NaN,Benign,NaN
1,1.533043e+09,Cf3cHf4jZr9nvD808i,192.168.100.113,123,147.231.100.5,123,udp,NaN,0.001741,48.0,...,NaN,0,Dd,1,76,1,76,NaN,Benign,NaN
2,1.533043e+09,CJgmSt3bSY6XwE9fzc,192.168.100.113,123,31.31.74.35,123,udp,NaN,0.004495,48.0,...,NaN,0,Dd,1,76,1,76,NaN,Benign,NaN
3,1.533043e+09,Cav32m4csR3OZYhShj,192.168.100.113,123,147.251.48.140,123,udp,NaN,0.006988,48.0,...,NaN,0,Dd,1,76,1,76,NaN,Benign,NaN
4,1.533043e+09,ClwPfA40tU9UT4nksg,192.168.100.113,123,147.231.100.5,123,udp,NaN,0.001487,48.0,...,NaN,0,Dd,1,76,1,76,NaN,Benign,NaN


## 3. Initial Dataset Check

In [4]:
print("Columns:")
print(df_raw.columns.tolist())

print("\nLabel counts:")
print(df_raw["label"].value_counts(dropna=False))

print("\nMissing values by column:")
display(df_raw.isna().sum().sort_values(ascending=False))


Columns:
['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes', 'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'tunnel_parents', 'label', 'detailed-label']

Label counts:
label
Malicious    8222
Benign       2181
Name: count, dtype: int64

Missing values by column:


,0
local_resp,10403
local_orig,10403
service,10403
tunnel_parents,10403
duration,6185
orig_bytes,6185
resp_bytes,6185
detailed-label,2181
proto,0
id.resp_h,0


## 4. Preprocess and Engineer Features

The target label is:

- `Benign` = 0  
- `Malicious` = 1  

The notebook drops identifiers that are not useful for general model learning, such as the unique connection ID and raw IP addresses. This keeps the project cleaner and helps avoid overfitting to specific addresses.


In [5]:
def preprocess_base_dataframe(df):
    df = df.copy()

    # Keep only rows with a valid label
    df = df[df["label"].isin(["Benign", "Malicious"])].copy()

    # Convert target to numeric
    df["target"] = df["label"].map({"Benign": 0, "Malicious": 1})

    # Columns that should be numeric if present
    numeric_candidates = [
        "ts", "id.orig_p", "id.resp_p", "duration", "orig_bytes", "resp_bytes",
        "missed_bytes", "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"
    ]

    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Feature engineering from the Mid-Term work
    if {"orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts"}.issubset(df.columns):
        df["total_bytes"] = df["orig_bytes"].fillna(0) + df["resp_bytes"].fillna(0)
        df["total_packets"] = df["orig_pkts"].fillna(0) + df["resp_pkts"].fillna(0)
        df["bytes_per_packet"] = df["total_bytes"] / (df["total_packets"] + 1)

    if {"orig_pkts", "resp_pkts", "duration"}.issubset(df.columns):
        df["packet_density"] = (
            df["orig_pkts"].fillna(0) + df["resp_pkts"].fillna(0)
        ) / (df["duration"].fillna(0) + 1)

    # Drop columns not used as model features
    columns_to_drop = [
        "label", "detailed-label", "target",
        "uid", "ts",
        "id.orig_h", "id.resp_h"
    ]

    # Drop columns that are entirely empty
    all_empty_cols = [col for col in df.columns if df[col].isna().all()]
    columns_to_drop.extend(all_empty_cols)

    X = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors="ignore")
    y = df["target"]

    return df, X, y

df_clean_base, X, y = preprocess_base_dataframe(df_raw)

print("Clean base dataset shape:", df_clean_base.shape)
print("Feature matrix shape:", X.shape)
print("Target distribution:")
print(y.value_counts().rename(index={0: "Benign", 1: "Malicious"}))

display(X.head())


Clean base dataset shape: (10403, 28)
Feature matrix shape: (10403, 17)
Target distribution:
target
Malicious    8222
Benign       2181
Name: count, dtype: int64


,id.orig_p,id.resp_p,proto,duration,orig_bytes,resp_bytes,conn_state,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,total_bytes,total_packets,bytes_per_packet,packet_density
0,123,123,udp,0.005490,48.0,48.0,SF,0,Dd,1,76,1,76,96.0,2,32.0,1.989080
1,123,123,udp,0.001741,48.0,48.0,SF,0,Dd,1,76,1,76,96.0,2,32.0,1.996524
2,123,123,udp,0.004495,48.0,48.0,SF,0,Dd,1,76,1,76,96.0,2,32.0,1.991050
3,123,123,udp,0.006988,48.0,48.0,SF,0,Dd,1,76,1,76,96.0,2,32.0,1.986121
4,123,123,udp,0.001487,48.0,48.0,SF,0,Dd,1,76,1,76,96.0,2,32.0,1.997030


### Export the cleaned pre-modeling dataset

In [6]:
clean_export = df_clean_base.copy()
clean_export.to_csv("ITAI1371_Final_Preprocessed_Dataset.csv", index=False)

print("Exported: ITAI1371_Final_Preprocessed_Dataset.csv")


Exported: ITAI1371_Final_Preprocessed_Dataset.csv


## 5. Split into Train, Validation, and Test Sets

In [7]:
# First split: 70% train and 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

# Second split: split temporary set equally into validation and test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train size:", X_train.shape[0], f"({X_train.shape[0] / len(X):.1%})")
print("Validation size:", X_valid.shape[0], f"({X_valid.shape[0] / len(X):.1%})")
print("Test size:", X_test.shape[0], f"({X_test.shape[0] / len(X):.1%})")

print("\nTrain label distribution:")
print(y_train.value_counts(normalize=True).rename(index={0: "Benign", 1: "Malicious"}))

print("\nValidation label distribution:")
print(y_valid.value_counts(normalize=True).rename(index={0: "Benign", 1: "Malicious"}))

print("\nTest label distribution:")
print(y_test.value_counts(normalize=True).rename(index={0: "Benign", 1: "Malicious"}))


Train size: 7282 (70.0%)
Validation size: 1560 (15.0%)
Test size: 1561 (15.0%)

Train label distribution:
target
Malicious    0.790305
Benign       0.209695
Name: proportion, dtype: float64

Validation label distribution:
target
Malicious    0.790385
Benign       0.209615
Name: proportion, dtype: float64

Test label distribution:
target
Malicious    0.790519
Benign       0.209481
Name: proportion, dtype: float64


### Export split files for submission/reference

In [8]:
train_export = X_train.copy()
train_export["target"] = y_train
train_export["label"] = train_export["target"].map({0: "Benign", 1: "Malicious"})

valid_export = X_valid.copy()
valid_export["target"] = y_valid
valid_export["label"] = valid_export["target"].map({0: "Benign", 1: "Malicious"})

test_export = X_test.copy()
test_export["target"] = y_test
test_export["label"] = test_export["target"].map({0: "Benign", 1: "Malicious"})

train_export.to_csv("ITAI1371_Final_Train.csv", index=False)
valid_export.to_csv("ITAI1371_Final_Validation.csv", index=False)
test_export.to_csv("ITAI1371_Final_Test.csv", index=False)

print("Exported:")
print("- ITAI1371_Final_Train.csv")
print("- ITAI1371_Final_Validation.csv")
print("- ITAI1371_Final_Test.csv")


Exported:
- ITAI1371_Final_Train.csv
- ITAI1371_Final_Validation.csv
- ITAI1371_Final_Test.csv


## 6. Build the Preprocessing Pipeline

In [9]:
numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

def make_one_hot_encoder(dense=False):
    """Handle scikit-learn versions that use sparse_output or sparse."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=not dense)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=not dense)

def make_preprocessor(dense=False):
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", make_one_hot_encoder(dense=dense))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features)
        ],
        remainder="drop"
    )

preprocessor = make_preprocessor(dense=False)
dense_preprocessor = make_preprocessor(dense=True)


Numeric features: ['id.orig_p', 'id.resp_p', 'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'total_bytes', 'total_packets', 'bytes_per_packet', 'packet_density']

Categorical features: ['proto', 'conn_state', 'history']


## 7. Define Models

In [10]:
model_definitions = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree Classifier": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting Classifier": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "K-Nearest Neighbors Classifier": KNeighborsClassifier(n_neighbors=5),
    "Support Vector Classifier": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

def make_pipeline(model):
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

models = {name: make_pipeline(model) for name, model in model_definitions.items()}

print("Models ready:")
for name in models:
    print("-", name)


Models ready:
- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier
- Gradient Boosting Classifier
- K-Nearest Neighbors Classifier
- Support Vector Classifier


## 8. Train and Evaluate Models

In [11]:
def evaluate_model(model, X_eval, y_eval, dataset_name):
    y_pred = model.predict(X_eval)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_eval)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_eval)
    else:
        y_score = y_pred

    return {
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1-Score": f1_score(y_eval, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, y_score)
    }

all_results = []
fitted_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    fitted_models[name] = model

    valid_result = evaluate_model(model, X_valid, y_valid, "Validation")
    test_result = evaluate_model(model, X_test, y_test, "Test")

    valid_result["Model"] = name
    test_result["Model"] = name

    all_results.extend([valid_result, test_result])

results_df = pd.DataFrame(all_results)
results_df = results_df[["Model", "Dataset", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]]

display(results_df)


Training Logistic Regression...
Training Decision Tree Classifier...
Training Random Forest Classifier...
Training Gradient Boosting Classifier...
Training K-Nearest Neighbors Classifier...
Training Support Vector Classifier...


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,Validation,1.000000,1.000000,1.0,1.00000,1.000000
1,Logistic Regression,Test,0.998719,0.998382,1.0,0.99919,1.000000
2,Decision Tree Classifier,Validation,1.000000,1.000000,1.0,1.00000,1.000000
3,Decision Tree Classifier,Test,1.000000,1.000000,1.0,1.00000,1.000000
4,Random Forest Classifier,Validation,1.000000,1.000000,1.0,1.00000,1.000000
5,Random Forest Classifier,Test,0.998719,0.998382,1.0,0.99919,1.000000
6,Gradient Boosting Classifier,Validation,1.000000,1.000000,1.0,1.00000,1.000000
7,Gradient Boosting Classifier,Test,0.998719,0.998382,1.0,0.99919,1.000000
8,K-Nearest Neighbors Classifier,Validation,1.000000,1.000000,1.0,1.00000,1.000000
9,K-Nearest Neighbors Classifier,Test,0.998719,0.998382,1.0,0.99919,0.996942


### Validation comparison table

In [12]:
validation_results = results_df[results_df["Dataset"] == "Validation"].copy()
validation_results = validation_results.sort_values(by="F1-Score", ascending=False)

display(validation_results)


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,Validation,1.0,1.0,1.0,1.0,1.0
2,Decision Tree Classifier,Validation,1.0,1.0,1.0,1.0,1.0
4,Random Forest Classifier,Validation,1.0,1.0,1.0,1.0,1.0
6,Gradient Boosting Classifier,Validation,1.0,1.0,1.0,1.0,1.0
8,K-Nearest Neighbors Classifier,Validation,1.0,1.0,1.0,1.0,1.0
10,Support Vector Classifier,Validation,1.0,1.0,1.0,1.0,1.0


### Test comparison table

In [13]:
test_results = results_df[results_df["Dataset"] == "Test"].copy()
test_results = test_results.sort_values(by="F1-Score", ascending=False)

display(test_results)


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
3,Decision Tree Classifier,Test,1.000000,1.000000,1.0,1.00000,1.000000
1,Logistic Regression,Test,0.998719,0.998382,1.0,0.99919,1.000000
5,Random Forest Classifier,Test,0.998719,0.998382,1.0,0.99919,1.000000
7,Gradient Boosting Classifier,Test,0.998719,0.998382,1.0,0.99919,1.000000
9,K-Nearest Neighbors Classifier,Test,0.998719,0.998382,1.0,0.99919,0.996942
11,Support Vector Classifier,Test,0.998719,0.998382,1.0,0.99919,1.000000


## 9. Select Best 3 Models from Validation Results

In [14]:
best_3_model_names = validation_results.head(3)["Model"].tolist()

print("Best 3 models based on Validation F1-Score:")
for model_name in best_3_model_names:
    print("-", model_name)


Best 3 models based on Validation F1-Score:
- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier


## 10. Voting Classifier Ensemble

This replaces the assignment's wording of "Voting Regressor" because the project is classification.


In [15]:
voting_estimators = [(name, model_definitions[name]) for name in best_3_model_names]

voting_classifier = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", VotingClassifier(
        estimators=voting_estimators,
        voting="soft"
    ))
])

print("Training Voting Classifier using:", best_3_model_names)
voting_classifier.fit(X_train, y_train)

voting_valid = evaluate_model(voting_classifier, X_valid, y_valid, "Validation")
voting_test = evaluate_model(voting_classifier, X_test, y_test, "Test")

voting_valid["Model"] = "Voting Classifier - Best 3"
voting_test["Model"] = "Voting Classifier - Best 3"

voting_results = pd.DataFrame([voting_valid, voting_test])
voting_results = voting_results[["Model", "Dataset", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]]

display(voting_results)


Training Voting Classifier using: ['Logistic Regression', 'Decision Tree Classifier', 'Random Forest Classifier']


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Voting Classifier - Best 3,Validation,1.000000,1.000000,1.0,1.00000,1.0
1,Voting Classifier - Best 3,Test,0.998719,0.998382,1.0,0.99919,1.0


## 11. Bayesian Model — Gaussian Naive Bayes

Gaussian Naive Bayes is used here as the Bayesian classification model. It gives probability-based predictions and gives us something to compare against the Voting Classifier.


In [16]:
bayesian_model = Pipeline(steps=[
    ("preprocessor", dense_preprocessor),
    ("model", GaussianNB())
])

print("Training Gaussian Naive Bayes...")
bayesian_model.fit(X_train, y_train)

bayes_valid = evaluate_model(bayesian_model, X_valid, y_valid, "Validation")
bayes_test = evaluate_model(bayesian_model, X_test, y_test, "Test")

bayes_valid["Model"] = "Gaussian Naive Bayes"
bayes_test["Model"] = "Gaussian Naive Bayes"

bayesian_results = pd.DataFrame([bayes_valid, bayes_test])
bayesian_results = bayesian_results[["Model", "Dataset", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]]

display(bayesian_results)


Training Gaussian Naive Bayes...


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Gaussian Naive Bayes,Validation,1.000000,1.000000,1.0,1.00000,1.000000
1,Gaussian Naive Bayes,Test,0.998719,0.998382,1.0,0.99919,0.996942


## 12. Final Model Comparison Table

In [17]:
final_results_df = pd.concat([results_df, voting_results, bayesian_results], ignore_index=True)

# Round for cleaner display and report use
final_results_rounded = final_results_df.copy()
metric_cols = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
final_results_rounded[metric_cols] = final_results_rounded[metric_cols].round(4)

display(final_results_rounded)

final_results_rounded.to_csv("ITAI1371_Final_Model_Comparison.csv", index=False)

print("Exported: ITAI1371_Final_Model_Comparison.csv")


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,Validation,1.0000,1.0000,1.0,1.0000,1.0000
1,Logistic Regression,Test,0.9987,0.9984,1.0,0.9992,1.0000
2,Decision Tree Classifier,Validation,1.0000,1.0000,1.0,1.0000,1.0000
3,Decision Tree Classifier,Test,1.0000,1.0000,1.0,1.0000,1.0000
4,Random Forest Classifier,Validation,1.0000,1.0000,1.0,1.0000,1.0000
5,Random Forest Classifier,Test,0.9987,0.9984,1.0,0.9992,1.0000
6,Gradient Boosting Classifier,Validation,1.0000,1.0000,1.0,1.0000,1.0000
7,Gradient Boosting Classifier,Test,0.9987,0.9984,1.0,0.9992,1.0000
8,K-Nearest Neighbors Classifier,Validation,1.0000,1.0000,1.0,1.0000,1.0000
9,K-Nearest Neighbors Classifier,Test,0.9987,0.9984,1.0,0.9992,0.9969


Exported: ITAI1371_Final_Model_Comparison.csv


### Pivot table format similar to the professor's guideline spreadsheet

In [18]:
validation_table = final_results_rounded[final_results_rounded["Dataset"] == "Validation"].drop(columns=["Dataset"])
test_table = final_results_rounded[final_results_rounded["Dataset"] == "Test"].drop(columns=["Dataset"])

print("Validation Metrics")
display(validation_table)

print("\nTest Metrics")
display(test_table)

validation_table.to_csv("ITAI1371_Final_Validation_Model_Table.csv", index=False)
test_table.to_csv("ITAI1371_Final_Test_Model_Table.csv", index=False)

print("Exported:")
print("- ITAI1371_Final_Validation_Model_Table.csv")
print("- ITAI1371_Final_Test_Model_Table.csv")


Validation Metrics


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,1.0,1.0,1.0,1.0,1.0
2,Decision Tree Classifier,1.0,1.0,1.0,1.0,1.0
4,Random Forest Classifier,1.0,1.0,1.0,1.0,1.0
6,Gradient Boosting Classifier,1.0,1.0,1.0,1.0,1.0
8,K-Nearest Neighbors Classifier,1.0,1.0,1.0,1.0,1.0
10,Support Vector Classifier,1.0,1.0,1.0,1.0,1.0
12,Voting Classifier - Best 3,1.0,1.0,1.0,1.0,1.0
14,Gaussian Naive Bayes,1.0,1.0,1.0,1.0,1.0



Test Metrics


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
1,Logistic Regression,0.9987,0.9984,1.0,0.9992,1.0000
3,Decision Tree Classifier,1.0000,1.0000,1.0,1.0000,1.0000
5,Random Forest Classifier,0.9987,0.9984,1.0,0.9992,1.0000
7,Gradient Boosting Classifier,0.9987,0.9984,1.0,0.9992,1.0000
9,K-Nearest Neighbors Classifier,0.9987,0.9984,1.0,0.9992,0.9969
11,Support Vector Classifier,0.9987,0.9984,1.0,0.9992,1.0000
13,Voting Classifier - Best 3,0.9987,0.9984,1.0,0.9992,1.0000
15,Gaussian Naive Bayes,0.9987,0.9984,1.0,0.9992,0.9969


Exported:
- ITAI1371_Final_Validation_Model_Table.csv
- ITAI1371_Final_Test_Model_Table.csv


## 13. Confusion Matrix and Classification Report for the Best Test Model

In [19]:
best_test_model_name = test_table.sort_values(by="F1-Score", ascending=False).iloc[0]["Model"]
print("Best model on Test F1-Score:", best_test_model_name)

model_lookup = fitted_models.copy()
model_lookup["Voting Classifier - Best 3"] = voting_classifier
model_lookup["Gaussian Naive Bayes"] = bayesian_model

best_model = model_lookup[best_test_model_name]
y_best_pred = best_model.predict(X_test)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_best_pred)
display(pd.DataFrame(
    cm,
    index=["Actual Benign", "Actual Malicious"],
    columns=["Predicted Benign", "Predicted Malicious"]
))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_best_pred,
    target_names=["Benign", "Malicious"],
    zero_division=0
))


Best model on Test F1-Score: Decision Tree Classifier

Confusion Matrix:


,Predicted Benign,Predicted Malicious
Actual Benign,327,0
Actual Malicious,0,1234



Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00       327
   Malicious       1.00      1.00      1.00      1234

    accuracy                           1.00      1561
   macro avg       1.00      1.00      1.00      1561
weighted avg       1.00      1.00      1.00      1561



In [20]:
print("Columns used as model inputs:")
for col in X.columns:
    if "label" in col.lower() or "target" in col.lower():
        print(col)

Columns used as model inputs:


In [22]:
print("Rows in model input dataset:", len(X))
print("Duplicate input rows:", X.duplicated().sum())

Rows in model input dataset: 10403
Duplicate input rows: 5973


In [23]:
duplicate_check = X.copy()
duplicate_check["target"] = y

print("Rows in model dataset:", len(duplicate_check))
print("Duplicate rows including target:", duplicate_check.duplicated().sum())

Rows in model dataset: 10403
Duplicate rows including target: 5973


## 14. Short Report Notes

Use this section as a starting point for the written report.

### Problem Type

This is a binary classification problem because the target label has two classes: Benign and Malicious.

### Approach

The original labeled dataset was loaded and cleaned. Missing markers such as `-` and `(empty)` were treated as missing values. Numeric fields were converted to numeric data types. Several engineered features were added, including `total_bytes`, `total_packets`, `bytes_per_packet`, and `packet_density`.

The data was split into training, validation, and test sets using a 70% / 15% / 15% split. The validation set was used to compare models before final testing. The test set was reserved for the final model evaluation.

### Models Compared

The following models were trained and compared:

- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier
- Gradient Boosting Classifier
- K-Nearest Neighbors Classifier
- Support Vector Classifier
- Voting Classifier using the best three validation models
- Gaussian Naive Bayes as the Bayesian model

### Model Selection

The best model should be selected using the validation and test results. For cybersecurity-style classification, Recall for the Malicious class is especially important because missing malicious traffic may be worse than incorrectly flagging benign traffic.

### Voting Classifier Note

The assignment mentions a Voting Regressor, but the project is classification. Therefore, a Voting Classifier was used instead.
